# 01 — Nested Oscillator Dynamics

## Claim

The thesis claims that four concentric orbits (primes 2, 3, 5, 7) form a system where:
- Every outer orbit is **constituted by** the states of its inner orbits
- State change is **distributed** across all nesting levels
- No orbit returns to the same **state**, even though every orbit returns to the same angular **position**
- At the outermost orbit, the state IS the total inner configuration — coordinate identification fails

## Model

Four coupled oscillators with frequencies in prime ratio (2:3:5:7), where each outer oscillator's state is constituted by the inner oscillators' states at that moment. We track:
1. Angular position of each oscillator (does it return?)
2. Total state at each return (has it changed?)
3. How the state complexity increases from innermost to outermost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

### Four Nested Oscillators

We model four oscillators with angular frequencies proportional to primes 2, 3, 5, 7.
Each oscillator has an angular position $\theta_i(t)$.
The **state** of each oscillator includes not just its own position but the positions of all inner oscillators.

- Orbit 2 (innermost): $\theta_2(t) = \omega_2 t$, state = $\theta_2$
- Orbit 3: $\theta_3(t) = \omega_3 t$, state = $(\theta_3, \theta_2)$
- Orbit 5: $\theta_5(t) = \omega_5 t$, state = $(\theta_5, \theta_3, \theta_2)$
- Orbit 7 (outermost): $\theta_7(t) = \omega_7 t$, state = $(\theta_7, \theta_5, \theta_3, \theta_2)$

In [ ]:
# Base frequencies proportional to primes
primes = np.array([2, 3, 5, 7])
omega = 2 * np.pi * primes  # angular frequencies

# Time array — long enough to see several full cycles of the outermost orbit
t = np.linspace(0, 10, 10000)

# Angular positions (mod 2π)
theta = np.array([np.mod(omega[i] * t, 2 * np.pi) for i in range(4)])

# Visualise the four oscillations
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
labels = ['Orbit 2 (bilateral)', 'Orbit 3 (vertical)', 'Orbit 5 (radial)', 'Orbit 7 (developmental)']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for i, (ax, label, color) in enumerate(zip(axes, labels, colors)):
    ax.plot(t, theta[i], color=color, linewidth=0.5, alpha=0.7)
    ax.set_ylabel(f'θ_{primes[i]}', fontsize=12)
    ax.set_title(label, fontsize=11, loc='left')

axes[-1].set_xlabel('Time', fontsize=12)
fig.suptitle('Angular Positions of Four Nested Oscillators', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### State at Angular Return Points

For each orbit, find moments where the **angular position** returns to (approximately) the same value.
Then check: is the **total state** (including inner orbits) the same at those return points?

In [ ]:
def find_returns(theta_i, t, tolerance=0.05):
    """Find times where theta_i returns to approximately its initial value."""
    initial = theta_i[0]
    # Find zero crossings of (theta_i - initial) mod 2π
    diff = np.abs(theta_i - initial)
    diff = np.minimum(diff, 2 * np.pi - diff)  # circular distance
    returns = []
    in_return = False
    for idx in range(1, len(t)):
        if diff[idx] < tolerance and not in_return:
            returns.append(idx)
            in_return = True
        elif diff[idx] >= tolerance:
            in_return = False
    return np.array(returns)

# Find return indices for each orbit
returns = {}
for i, p in enumerate(primes):
    ret_idx = find_returns(theta[i], t)
    returns[p] = ret_idx
    print(f'Orbit {p}: {len(ret_idx)} returns to initial position')

In [ ]:
def state_at_returns(orbit_idx, returns_idx, theta_all):
    """
    At each return of orbit_idx, record the state of all inner orbits.
    Inner orbits are those with index < orbit_idx.
    Returns array of inner states at each return point.
    """
    if len(returns_idx) == 0:
        return np.array([])
    # Inner orbits: indices 0..orbit_idx-1
    inner_states = theta_all[:orbit_idx+1, returns_idx].T  # shape: (n_returns, n_inner+1)
    return inner_states

# For each orbit, compute the total state at each return
print('State change at angular return points:')
print('=' * 60)
for i, p in enumerate(primes):
    ret_idx = returns[p]
    if len(ret_idx) < 2:
        print(f'Orbit {p}: insufficient returns')
        continue
    states = state_at_returns(i, ret_idx, theta)
    # Compute pairwise distances between consecutive return states
    diffs = []
    for j in range(1, len(states)):
        # Circular distance for each component
        d = states[j] - states[0]
        d = np.minimum(np.abs(d), 2*np.pi - np.abs(d))
        diffs.append(np.linalg.norm(d))
    diffs = np.array(diffs)
    print(f'\nOrbit {p} (contains {i+1} components):')
    print(f'  Returns found: {len(ret_idx)}')
    print(f'  Mean state distance from initial: {diffs.mean():.4f}')
    print(f'  Min state distance: {diffs.min():.4f}')
    print(f'  Max state distance: {diffs.max():.4f}')
    print(f'  Any exact return? {"YES" if diffs.min() < 0.01 else "NO"}')

### State Complexity by Nesting Level

Measure how much the total state varies at each orbit's return points.
The thesis predicts: complexity increases monotonically from innermost to outermost.

In [ ]:
# Measure state variance at return points for each orbit level
complexity = []
for i, p in enumerate(primes):
    ret_idx = returns[p]
    if len(ret_idx) < 2:
        complexity.append(0)
        continue
    states = state_at_returns(i, ret_idx, theta)
    # Variance of inner states at return points = measure of how much
    # "the same position" has different states
    var = np.var(states[:, :i], axis=0).sum() if i > 0 else 0
    complexity.append(var)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([f'Orbit {p}' for p in primes], complexity, color=colors)
ax.set_ylabel('State variance at angular returns')
ax.set_title('State Complexity by Nesting Level\n(variance of inner states when outer coordinate returns)')
plt.tight_layout()
plt.show()

print('\nComplexity gradient (innermost → outermost):')
for p, c in zip(primes, complexity):
    print(f'  Orbit {p}: {c:.4f}')

## Verdict

*(To be filled after running the computation)*